# 実習1：ペンギンの種を分類しよう

4つの測定値から、ペンギンの種（Adelie・Chinstrap・Gentoo）を予測します。

**今日の目標：データ確認 → 学習 → 予測 → 評価を一通り体験する。**

Google Colabで上から順に各コードセルの ▶ を押してください。CPUで実行できます。最初のデータ読み込みにはインターネット接続が必要です。

今回は**決定木（Decision Tree）**を使います。特徴量に対する質問を組み合わせて分類する方法です。自分で関数やクラスを定義する必要はありません。


## 1. データを読み込む

ライブラリは、便利な機能をまとめた道具です。`import`で使えるようにします。
`pd.read_csv()`は、CSVファイルを表形式のデータ（DataFrame）として読み込みます。

`head()`で最初の5行を見てみましょう。1行が1羽のペンギンです。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

url = "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv"
df = pd.read_csv(url)
print(df.head())

## 2. 使う列を選び、欠損値を確認する

| 列名 | 意味 |
|---|---|
| bill_length_mm | くちばしの長さ（mm） |
| bill_depth_mm | くちばしの厚み・深さ（mm） |
| flipper_length_mm | 翼の長さ（mm） |
| body_mass_g | 体重（g） |
| species | 種：正解ラベル |

`features`は、入力に使う4つの列名を入れたリストです。`df[列名のリスト]`で必要な列を取り出します。

欠損値（NaN）は、値が記録されていない部分です。今回は、**使う5列に欠損値がある行だけを除外**します。平均値などで補完する処理は行いません。


In [ ]:
features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
df = df[features + ["species"]]

print("欠損値の数：")
print(df.isnull().sum())

df = df.dropna()
print("使用するデータ数：", len(df))
print(df["species"].value_counts())

## 3. データを図で見る

横軸は翼の長さ、縦軸は体重です。色とマーカーの形で種を区別します。

**問い：分かれやすい種と、重なっている種はどれでしょうか？**

この図では2つの特徴量だけを見ますが、モデルは4つすべてを使います。図の軸は日本語フォントの設定が不要な英語表記です。


In [ ]:
sns.scatterplot(data=df, x="flipper_length_mm", y="body_mass_g", hue="species", style="species")
plt.xlabel("Flipper length (mm)")
plt.ylabel("Body mass (g)")
plt.show()

## 4. 入力 X と正解 y を分ける

- `X`：4つの測定値。モデルへの入力です。
- `y`：種の名前。答え合わせに使う正解です。

**speciesをXに入れないことが重要です。** 種の名前は文字列のまま扱えます。


In [ ]:
X = df[features]
y = df["species"]

print("入力 X：")
print(X.head())
print("正解 y：")
print(y.head())

## 5. 学習用とテスト用に分割する

`train_test_split()`で、約80%を学習用、約20%をテスト用に分けます。

| 設定 | 意味 |
|---|---|
| test_size=0.2 | 約20%をテスト用にする |
| random_state=42 | 同じデータと環境で分割を再現しやすくする |
| stratify=y | 各種の割合をできるだけ保って分割する |

`42`は固定用の番号で、正答率を上げる特別な数字ではありません。テスト用にも正解はありますが、学習には使いません。


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("学習用：", len(X_train))
print("テスト用：", len(X_test))

## 6. 決定木を学習する

`DecisionTreeClassifier()`でモデルを用意し、`fit()`で学習します。

`max_depth=3`は、質問を分岐させる木の深さを最大3にする設定です。今回は最初からこの設定に固定します。
決定木では通常、単位をそろえるための標準化は必要ありません。


In [ ]:
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)

print("学習が終わりました。")

## 7. テストデータを予測し、正答率を求める

`predict()`にはテスト用の入力だけを渡します。
`accuracy_score()`で正解と予測を比較します。

**正答率 = 正しく予測したテストデータ数 ÷ テストデータ数**

`round(数値, 1)`は小数第1位までの表示に使います。


In [ ]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("正答率：", round(accuracy * 100, 1), "%")

## 8. 一つずつ答え合わせをする

`pd.DataFrame()`で、正解と予測を並べた表を作ります。
`==`は左右が等しいかを調べます。正解ならTrue、間違いならFalseです。

`result[result["正解"] == False]`は、間違えた行だけを取り出す書き方です。
元データの行番号を残しているため、番号は連番でないことがあります。


In [ ]:
result = pd.DataFrame({"実際の種": y_test, "予測した種": y_pred})
result["正解"] = result["実際の種"] == result["予測した種"]

print(result.head(10))
print("間違えたデータ：")
print(result[result["正解"] == False])

## 9. 新しい1羽を予測する

説明用の仮想データです。くちばしの長さ48 mm、深さ15 mm、翼の長さ215 mm、体重5,000 gを入力します。

学習時と**同じ列名・同じ順序・同じ単位**で渡します。`[[...]]`は「1行分の表」を作るための書き方です。
`[0]`で、予測結果の最初の1件を取り出します。

この1羽には確認済みの正解がないため、予測はできますが、この結果だけでは正答率を計算できません。


In [ ]:
new_penguin = pd.DataFrame([[48, 15, 215, 5000]], columns=features)
prediction = model.predict(new_penguin)

print("予測した種：", prediction[0])

## 確認問題

1. 入力に使った4つの特徴量と、正解ラベルを答えてください。
2. 学習用とテスト用のデータ数、テストの正答率を書いてください。
3. 間違えたデータでは、どの種をどの種と予測していましたか？
4. `model.predict(X_test)`に`y_test`を渡さないのはなぜですか？

**覚えたい3行**

```python
model.fit(X_train, y_train)       # 学習
y_pred = model.predict(X_test)   # 予測
accuracy_score(y_test, y_pred)    # 評価
```

学習した規則が正解と完全に一致するとは限りません。テスト結果を何度も見ながら設定を選ぶと、公平な評価が難しくなります。モデルや設定の比較は、別の検証データなどを使う次の段階で扱います。

## 参考資料

- [Palmer Penguins：データと出典](https://github.com/allisonhorst/palmerpenguins)
- [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
- [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)
- [accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)
